In [24]:
import os
import sys
sys.path.append(os.path.abspath('..'))
from lib.Forecast.SimpleTransformerForecast import SimpleTransformerForecast
from lib.Forecast.PatchTransformerForecast import PatchTransformerForecast, PatchTransformerForecast2
from lib.Forecast.PatchTransformerExoForecast import PatchTransformerExoForecast
from lib.Dataloaders.ECGDataset import ECGDataset
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import torch
import numpy as np
import torch.nn.functional as F
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader
import os
import time
import warnings
warnings.filterwarnings("ignore")

In [6]:
# Function to train the model
def train_loop(model,
               train,
               val,
               optimizer,
               scheduler=None,
               patience=5,
               epochs=100,
               exogenous=False,
               conditional=False,
               lossf=F.mse_loss):
    """_Training loop for the model_

    Args:
        model: model to train
        optimizer: pytorch optimizer, for example torch.optim.Adam
        train: training data
        val: validation data
        epochs: number of epochs

    Returns:
        _type_: training history, a dictionary with the training and validation loss for each epoch
    """

    def epoch_loss(dataset, conditional=conditional, exogenous=exogenous):
        data_loss = 0.0
        for i, data in enumerate(dataset):
            if conditional:
                lookback_data, horizon_data, cond_data = data
                cond_data = cond_data.to('cuda')
            else:
                lookback_data, horizon_data = data
                cond_data = None
            lookback_data = lookback_data.to('cuda')
            horizon_data = horizon_data.to('cuda')
            # We do not have the exogenous data in the dataset, so we will use the lookback data as exogenous data
            if exogenous:
                exogenous_data = lookback_data.clone()
            exogenous_data = exogenous_data.to('cuda') if exogenous else None
            inputs = lookback_data.to('cuda')
            y = horizon_data.to('cuda')
            outputs = model(inputs,
                            exo=exogenous_data if exogenous else None,
                            cond=cond_data if conditional else None).squeeze()

            loss = lossf(y, outputs)
            data_loss += loss.item()
        return data_loss / (i + 1)

    def early_stopping(val_loss, patience=5):
        if len(val_loss) > patience:
            if val_loss[-1] > np.mean(val_loss[-(patience + 1):-1]):
                return True

    hist_loss = {'train': [], 'val': []}
    pbar = tqdm(range(epochs))
    for epoch in pbar:  # loop for all the epochs
        # print(f"Epoch {epoch + 1}/{epochs}")
        for i, data in enumerate(train):
            if conditional:
                lookback_data, horizon_data, cond_data = data
                cond_data = cond_data.to('cuda')
            else:
                lookback_data, horizon_data = data
                cond_data = None
            lookback_data = lookback_data.to('cuda')
            horizon_data = horizon_data.to('cuda')
            # We do not have the exogenous data in the dataset, so we will use the lookback data as exogenous data
            if exogenous:
                exogenous_data = lookback_data.clone()
            exogenous_data = exogenous_data.to('cuda') if exogenous else None

            # take the data and upload it to the GPU
            inputs = lookback_data.to('cuda')
            y = horizon_data.to('cuda')

            # Reset the gradients
            optimizer.zero_grad()

            # Apply the data to the model
            outputs = model(inputs,
                            exo=exogenous_data if exogenous else None,
                            cond=cond_data if conditional else None).squeeze()
            # Calculate the loss
            loss = lossf(y, outputs)

            # Make the backward pass
            loss.backward()
            optimizer.step()

        if scheduler is not None:
            scheduler.step()

        # Calculate the loss in the training and validation sets
        with torch.no_grad():
            hist_loss['train'].append(
                epoch_loss(train, conditional=conditional,
                           exogenous=exogenous))
            hist_loss['val'].append(
                epoch_loss(val, conditional=conditional, exogenous=exogenous))

        # Show the loss in the training and validation sets
        pbar.set_postfix({
            'train': hist_loss['train'][-1],
            'val': hist_loss['val'][-1],
            'lr': optimizer.param_groups[0]['lr']
        })

        # If the loss in the validation set does not decrease, stop the training
        if early_stopping(hist_loss['val'], patience):
            break

    return hist_loss

In [7]:
def plot_prediction(model, data, channel=0):
    X, y = data

    pred = model(torch.tensor(X).unsqueeze(0).to(
        'cuda')).squeeze().cpu().detach().numpy()

    plt.figure(figsize=(12, 6))
    ax = plt.subplot(1, 2, 1)
    plt.plot(X[channel], label='lookback')
    mx, mn = np.max(X[channel]), np.min(X[channel])
    plt.legend()
    ax = plt.subplot(1, 2, 2)
    plt.plot(y, label='true')
    plt.plot(pred, label='pred')
    #plt.ylim(mn , mx )
    plt.legend()

In [8]:
lookback = 100
horizon = 25
batch_size = 2048
stride = 10
norm = "zscore"
norm_all = False
samples = (2500, 200, 200)
channel = 0
labels = True

In [9]:
train = ECGDataset(dir='../Preprocess/PTBXL',
                   dataset='train',
                   nsamples=samples[0],
                   channel=channel,
                   norm=norm,
                   norm_all=norm_all,
                   lookback=lookback,
                   horizon=horizon,
                   stride=stride,
                   labels=labels)
val = ECGDataset(dir='../Preprocess/PTBXL',
                 dataset='validation',
                 nsamples=samples[1],
                 channel=channel,
                 norm=norm,
                 norm_all=norm_all,
                 lookback=lookback,
                 horizon=horizon,
                 stride=stride,
                 labels=labels)
test = ECGDataset(dir='../Preprocess/PTBXL',
                   dataset='test',
                   nsamples=samples[2],
                   channel=channel,
                   norm=norm,
                   norm_all=norm_all,
                   lookback=lookback,
                   horizon=horizon,
                   stride=stride,
                   labels=labels)
train = DataLoader(train, batch_size=batch_size, shuffle=True)
val = DataLoader(val, batch_size=batch_size, shuffle=False)
test = DataLoader(test, batch_size=batch_size, shuffle=False)

Loading data from ../Preprocess/PTBXL
train
X_train shape is (220000, 8, 100)
y_train shape is (220000, 25)
dlabels shape is (88,)
NORM=zscore
Loading data from ../Preprocess/PTBXL
validation
X_train shape is (17600, 8, 100)
y_train shape is (17600, 25)
dlabels shape is (88,)
NORM=zscore
Loading data from ../Preprocess/PTBXL
test
X_train shape is (17600, 8, 100)
y_train shape is (17600, 25)
dlabels shape is (88,)
NORM=zscore


In [ ]:
# model = SimpleTransformerForecast(lookback=lookback,
#                                   horizon=horizon,
#                                   input_dim=8,
#                                   target_dim=1,
#                                   d_model=16,
#                                   n_heads=2,
#                                   num_layers=6, dropout=0).to('cuda')
# model.compile()

In [ ]:
# model = PatchTransformerForecast(lookback=lookback,
#                                 horizon=horizon,
#                                 input_dim=8,
#                                 target_dim=1,
#                                 d_model=64,
#                                 n_heads=8,
#                                 num_layers=2, dropout=0, patch_len=10).to('cuda')

In [ ]:
# model = PatchTransformerForecast2(lookback=lookback,
#                                   horizon=horizon,
#                                   input_dim=8,
#                                   target_dim=1,
#                                   d_model=64,
#                                   n_heads=8,
#                                   num_layers=2,
#                                   norm_first=True,
#                                   dropout=0,
#                                   patch_len=10,
#                                   layer_norm_eps=1e-5,
#                                   bias=True,
#                                   swiglu=True,
#                                   rmsnorm=True,
#                                   trans_norm=True,
#                                   device='cuda').to('cuda')
# model.compile()

In [ ]:
model = PatchTransformerExoForecast(lookback=lookback,
                                  horizon=horizon,
                                  input_dim=8,
                                  target_dim=1,
                                  condition_dim=8,
                                  d_model=64,
                                  n_heads=8,
                                  num_layers=2,
                                  norm_first=True,
                                  dropout=0,
                                  patch_len=10,
                                  layer_norm_eps=1e-5,
                                  bias=True,
                                  swiglu=True,
                                  rmsnorm=True,
                                  trans_norm=True,
                                  device='cuda', verbose=False).to('cuda')
#model.compile() # model.compile serve per inizializzare correttamente il modello prima di caricare i pesi

In [11]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = StepLR(optimizer, step_size=20, gamma=0.90)

In [ ]:
# # Testing the model with random input
# random_input = torch.randn(32, 8, 120).to('cuda')
# model.to('cuda')
# model(random_input).shape

In [ ]:
hist_loss = train_loop(model, train, val, optimizer, scheduler, epochs=1000, patience=100, exogenous=True, conditional=True)

In [ ]:
plt.plot(hist_loss['train'], label='train')
plt.plot(hist_loss['val'], label='val')
plt.legend()


In [ ]:
print(scheduler.get_last_lr())

In [ ]:
plot_prediction(model, train.dataset[0])

In [ ]:
plot_prediction(model, val.dataset[0])

In [ ]:
plot_prediction(model, test.dataset[0])

In [ ]:
def dataset_loss(dataset):
    data_loss = 0.0
    for i, (lookback_data, horizon_data) in enumerate(dataset):
        inputs = lookback_data.to('cuda')
        y = horizon_data.to('cuda')
        outputs = model(inputs).squeeze()
        loss = F.mse_loss(y, outputs)
        data_loss += loss.item()
    return data_loss / (i + 1)
print('Train loss:', dataset_loss(train))
print('Val loss:', dataset_loss(val))
print('Test loss:', dataset_loss(test))

In [ ]:
# Dopo che l'addestramento è finito e sei soddisfatto
percorso_salvataggio = 'pesi_ecg_patch.pth'

# Salviamo SOLO lo state_dict (i pesi)
torch.save(model.state_dict(), percorso_salvataggio)
print(f"[*] Pesi salvati correttamente in: {percorso_salvataggio}")

In [12]:
percorso_pesi = "weights_transformer_patch.pth"
try:
    model.load_state_dict(torch.load(percorso_pesi, map_location='cpu', weights_only=True))
    print(f"[*] Pesi del modello '{percorso_pesi}' caricati con successo!")
except FileNotFoundError:
    print(f"[!] ATTENZIONE: File '{percorso_pesi}' non trovato nella cartella attuale.")
    print(f"[!] Il modello eseguirà l'inferenza con pesi casuali (inizializzazione dummy per stress test).")

# 3. Impostiamo il modello in modalità valutazione (disattiva dropout)
model.eval()

[*] Pesi del modello 'weights_transformer_patch.pth' caricati con successo!


PatchTransformerExoForecast(
  (patch_projection): Conv1d(80, 64, kernel_size=(1,), stride=(1,))
  (patch_exogenous_projection): Conv1d(80, 64, kernel_size=(1,), stride=(1,))
  (patch_condition_projection): Linear(in_features=8, out_features=64, bias=True)
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiHeadAttention(
          (packed_proj): Linear(in_features=64, out_features=192, bias=True)
          (out_proj): Linear(in_features=64, out_features=64, bias=True)
        )
        (linear1): PackedSwiGLUFFN(
          (w13): Linear(in_features=64, out_features=340, bias=False)
          (w2): Linear(in_features=170, out_features=256, bias=False)
        )
        (dropout): Dropout(p=0, inplace=False)
        (linear2): Linear(in_features=256, out_features=64, bias=True)
        (norm1): RMSNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): RMSNorm((64,), eps=1e-05, elementwise_af

In [23]:
import torch
from torchinfo import summary

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

batch = next(iter(train))
if len(batch) == 3:
    real_lookback, real_horizon, real_cond = batch
else:
    real_lookback, real_horizon = batch
    real_cond = real_lookback[:, :, -1]

real_lookback = real_lookback.float().to(device)
real_exo = real_lookback.clone()
real_cond = real_cond.float().to(device)

print("devices:", real_lookback.device, real_exo.device, real_cond.device, next(model.parameters()).device)

model.eval()
with torch.no_grad():
    out = model(real_lookback, exo=real_exo, cond=real_cond)
    print("output shape:", tuple(out.shape))

summary(
    model,
    input_data=(real_lookback,),
    exo=real_exo,
    cond=real_cond,
    device=device,
    depth=4,
    mode="eval",
    col_names=["input_size", "output_size", "num_params"],
)

devices: cuda:0 cuda:0 cuda:0 cuda:0
output shape: (2048, 25)


Layer (type:depth-idx)                        Input Shape               Output Shape              Param #
PatchTransformerExoForecast                   [2048, 8, 100]            [2048, 25]                --
├─Conv1d: 1-1                                 [2048, 80, 10]            [2048, 64, 10]            5,184
├─Conv1d: 1-2                                 [2048, 80, 10]            [2048, 64, 10]            5,184
├─Linear: 1-3                                 [2048, 1, 8]              [2048, 1, 64]             576
├─TransformerEncoder: 1-4                     [2048, 21, 64]            [2048, 21, 64]            --
│    └─ModuleList: 2-1                        --                        --                        --
│    │    └─TransformerEncoderLayer: 3-1      [2048, 21, 64]            [2048, 21, 64]            --
│    │    │    └─RMSNorm: 4-1                 [2048, 21, 64]            [2048, 21, 64]            64
│    │    │    └─MultiHeadAttention: 4-2      [2048, 21, 64]            [2048, 